<div style="background-image: url('https://www.dropbox.com/scl/fi/nwtuyf23tzyuq9gfan3oc/mcnair.jpg?rlkey=4t1of5cy6bfgxn6xck7yu9hvo&dl=1'); background-size: cover; background-position: center; height: 300px; display: flex; align-items: center; justify-content: center; color: white; text-shadow: 2px 2px 4px rgba(0,0,0,0.7); margin-bottom: 20px; position: relative;">
  <h1 style="text-align: center; font-size: 2.5em; margin: 0;">JGSB Python Workshop <br> Part 13: Capstone Project<br>Neural Networks & Digit Recognition</h1>
  <div style="position: absolute; bottom: 10px; left: 15px; font-size: 0.9em; color: white; text-shadow: 2px 2px 4px rgba(0,0,0,0.7);">
    Authored by Kerry Back
  </div>
  <div style="position: absolute; bottom: 10px; right: 15px; text-align: right; font-size: 0.9em; color: white; text-shadow: 2px 2px 4px rgba(0,0,0,0.7);">
    Rice University, 2025
  </div>
</div>

# Welcome to the Capstone Project!

Congratulations on making it to the final part of the Python workshop! In this capstone project, you'll apply everything you've learned to build and train a **neural network** that can recognize handwritten digits.

## What You'll Accomplish

By the end of this project, you will:

1. **Understand Neural Networks**: Learn how multi-layer perceptrons work from the ground up
2. **Build from Scratch**: Implement forward propagation and backpropagation manually
3. **Train a Real Model**: Use your network to classify handwritten digits (0-9)
4. **Visualize Results**: See your model's predictions and understand where it succeeds and fails
5. **Apply All Your Skills**: Use NumPy, Pandas, Matplotlib, and optimization techniques

## Skills Integration

This project brings together everything from the workshop:
- **NumPy**: Matrix operations and mathematical computations
- **Pandas**: Data handling and analysis
- **Matplotlib**: Data visualization and results interpretation
- **Functions**: Modular code design and reusability
- **Flow Control**: Loops and conditionals for training algorithms
- **Optimization**: Gradient descent and parameter tuning

Let's start by understanding neural networks with a simple, intuitive example!

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducible results
np.random.seed(42)

print("🚀 Capstone Project - Neural Networks Ready!")
print("All libraries imported successfully.")

## Understanding Neural Networks: A Simple Example

Before diving into digit recognition, let's understand how neural networks work using a simple example. We'll build a network that can classify whether a customer will make a purchase based on two factors: **age** and **income**.

### The Business Problem
Imagine you're analyzing customer data and want to predict purchase behavior. You have:
- **Input 1 (x₁)**: Customer age (normalized)
- **Input 2 (x₂)**: Customer income (normalized) 
- **Target**: 1 if customer purchases, 0 if not

For our example, we'll use a simple rule: customers purchase when `age + income ≥ 0` (after normalization).

In [ ]:
# Generate sample customer data
np.random.seed(42)
n_customers = 200

# Generate random age and income (normalized to range [-2, 2])
x1_age = np.random.uniform(-2, 2, n_customers)
x2_income = np.random.uniform(-2, 2, n_customers)

# True target: purchase if age + income >= 0 (with some noise)
noise = np.random.normal(0, 0.3, n_customers)  # Add realistic noise
true_score = x1_age + x2_income + noise
y_purchase = (true_score >= 0).astype(int)

# Create DataFrame for easy viewing
customer_data = pd.DataFrame({
    'age_normalized': x1_age,
    'income_normalized': x2_income, 
    'purchase': y_purchase
})

print("Customer Data Sample:")
print(customer_data.head(10))

# Visualize the data
plt.figure(figsize=(10, 8))
colors = ['red', 'blue']
labels = ['No Purchase', 'Purchase']

for i in [0, 1]:
    mask = y_purchase == i
    plt.scatter(x1_age[mask], x2_income[mask], c=colors[i], alpha=0.6, 
               label=labels[i], s=50)

# Draw the true decision boundary
x_line = np.linspace(-2, 2, 100)
y_line = -x_line  # age + income = 0 => income = -age
plt.plot(x_line, y_line, 'k--', linewidth=2, label='True Boundary (age + income = 0)')

plt.xlabel('Age (normalized)')
plt.ylabel('Income (normalized)')
plt.title('Customer Purchase Behavior\n(Target: Purchase if Age + Income ≥ 0)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nData Summary:")
print(f"Total customers: {n_customers}")
print(f"Customers who purchased: {np.sum(y_purchase)} ({np.mean(y_purchase):.1%})")
print(f"Customers who didn't purchase: {n_customers - np.sum(y_purchase)} ({1-np.mean(y_purchase):.1%})")

## Multi-Layer Perceptron Architecture

Now let's build a neural network to learn this pattern. Our network will have:

- **Input Layer**: 2 neurons (age, income)
- **Hidden Layer**: 3 neurons (to learn complex patterns)
- **Output Layer**: 1 neuron (probability of purchase)

### Network Structure
```
Input Layer    Hidden Layer    Output Layer
     x₁ ────────── h₁ ────────────
              ×  ╱  ╲  ×          ╲
     x₂ ────────── h₂ ──────────── ŷ
              ×  ╲  ╱  ×          ╱
              ──── h₃ ────────────
```

### Key Concepts
- **Weights (W)**: Control the strength of connections between neurons
- **Biases (b)**: Allow neurons to fire even when inputs are zero
- **Activation Function**: Sigmoid function to convert outputs to probabilities
- **Forward Propagation**: Data flows from input to output
- **Backpropagation**: Errors flow backward to update weights

In [ ]:
# Define the sigmoid activation function
def sigmoid(z):
    """Sigmoid activation function: converts any real number to (0,1)"""
    # Clip z to prevent overflow
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    """Derivative of sigmoid function (needed for backpropagation)"""
    s = sigmoid(z)
    return s * (1 - s)

# Test the sigmoid function
z_values = np.linspace(-10, 10, 100)
sigmoid_values = sigmoid(z_values)

plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
plt.plot(z_values, sigmoid_values, 'b-', linewidth=2)
plt.title('Sigmoid Activation Function')
plt.xlabel('Input (z)')
plt.ylabel('Output σ(z)')
plt.grid(True, alpha=0.3)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.7, label='Decision Threshold')
plt.legend()

plt.subplot(1, 2, 2)
derivative_values = sigmoid_derivative(z_values)
plt.plot(z_values, derivative_values, 'g-', linewidth=2)
plt.title('Sigmoid Derivative')
plt.xlabel('Input (z)')
plt.ylabel("σ'(z)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Sigmoid Function Properties:")
print(f"• Range: (0, 1) - perfect for probabilities")
print(f"• σ(0) = {sigmoid(0):.3f} - neutral point")
print(f"• σ(large positive) ≈ 1 - high confidence 'yes'")
print(f"• σ(large negative) ≈ 0 - high confidence 'no'")
print(f"• Smooth and differentiable - essential for gradient descent")

## Manual Forward Propagation Example

Let's walk through a single example step by step to understand how the network makes predictions. We'll use initial weights and biases, then show how the calculation flows through the network.

In [ ]:
# Initialize network parameters (we'll start with random values)
# These will be learned during training

# Weights from input layer (2 neurons) to hidden layer (3 neurons)
W1 = np.array([
    [0.5, -0.2],   # Hidden neuron 1 weights: [age_weight, income_weight]
    [-0.3, 0.8],   # Hidden neuron 2 weights
    [0.1, 0.4]     # Hidden neuron 3 weights
])

# Biases for hidden layer (3 neurons)
b1 = np.array([0.1, -0.5, 0.2])

# Weights from hidden layer (3 neurons) to output layer (1 neuron)
W2 = np.array([[0.7], [-0.4], [0.9]])

# Bias for output layer (1 neuron)
b2 = np.array([0.3])

print("Initial Network Parameters:")
print(f"\nInput to Hidden Weights (W1):")
print(f"  Hidden 1: age={W1[0,0]:5.2f}, income={W1[0,1]:5.2f}")
print(f"  Hidden 2: age={W1[1,0]:5.2f}, income={W1[1,1]:5.2f}")
print(f"  Hidden 3: age={W1[2,0]:5.2f}, income={W1[2,1]:5.2f}")

print(f"\nHidden Layer Biases (b1): {b1}")

print(f"\nHidden to Output Weights (W2): {W2.flatten()}")
print(f"Output Bias (b2): {b2[0]}")

# Let's trace through a specific example
example_customer = np.array([1.2, -0.8])  # Age=1.2, Income=-0.8
print(f"\n" + "="*50)
print(f"FORWARD PROPAGATION EXAMPLE")
print(f"="*50)
print(f"Input Customer: Age={example_customer[0]}, Income={example_customer[1]}")

# Step 1: Input to Hidden Layer
print(f"\nStep 1: Input → Hidden Layer")
z1 = W1 @ example_customer + b1  # Matrix multiplication + bias
a1 = sigmoid(z1)  # Apply activation function

for i in range(3):
    calculation = f"{W1[i,0]:.2f}×{example_customer[0]:.1f} + {W1[i,1]:.2f}×{example_customer[1]:.1f} + {b1[i]:.2f}"
    print(f"  Hidden {i+1}: {calculation} = {z1[i]:.3f} → σ({z1[i]:.3f}) = {a1[i]:.3f}")

# Step 2: Hidden to Output Layer
print(f"\nStep 2: Hidden → Output Layer")
z2 = W2.T @ a1 + b2  # Matrix multiplication + bias
a2 = sigmoid(z2)  # Apply activation function

calculation_parts = [f"{W2[i,0]:.2f}×{a1[i]:.3f}" for i in range(3)]
calculation = " + ".join(calculation_parts) + f" + {b2[0]:.2f}"
print(f"  Output: {calculation} = {z2[0]:.3f} → σ({z2[0]:.3f}) = {a2[0]:.3f}")

# Interpretation
print(f"\nPrediction Results:")
print(f"  Predicted probability of purchase: {a2[0]:.1%}")
prediction = 1 if a2[0] >= 0.5 else 0
confidence = a2[0] if prediction == 1 else (1 - a2[0])
print(f"  Classification: {'Purchase' if prediction == 1 else 'No Purchase'}")
print(f"  Confidence: {confidence:.1%}")

# Check against true rule
true_result = 1 if (example_customer[0] + example_customer[1]) >= 0 else 0
true_sum = example_customer[0] + example_customer[1]
print(f"\nActual Rule: Age + Income = {true_sum:.1f} → {'Purchase' if true_result == 1 else 'No Purchase'}")
print(f"Prediction Match: {'✓ Correct' if prediction == true_result else '✗ Incorrect'}")

## Building the Complete Neural Network Class

Now let's create a complete neural network class that can learn from data through backpropagation and gradient descent.

In [ ]:
class SimpleNeuralNetwork:
    def __init__(self, input_size=2, hidden_size=3, output_size=1, learning_rate=0.1):
        """
        Initialize a simple neural network
        
        Parameters:
        - input_size: Number of input features
        - hidden_size: Number of neurons in hidden layer
        - output_size: Number of output neurons
        - learning_rate: How fast the network learns
        """
        self.learning_rate = learning_rate
        
        # Initialize weights with small random values
        self.W1 = np.random.normal(0, 0.5, (hidden_size, input_size))
        self.b1 = np.random.normal(0, 0.1, (hidden_size,))
        self.W2 = np.random.normal(0, 0.5, (output_size, hidden_size))
        self.b2 = np.random.normal(0, 0.1, (output_size,))
        
        # Store training history
        self.loss_history = []
        self.accuracy_history = []
        
    def forward(self, X):
        """
        Forward propagation
        """
        # Input to hidden layer
        self.z1 = X @ self.W1.T + self.b1
        self.a1 = sigmoid(self.z1)
        
        # Hidden to output layer
        self.z2 = self.a1 @ self.W2.T + self.b2
        self.a2 = sigmoid(self.z2)
        
        return self.a2
    
    def compute_loss(self, y_true, y_pred):
        """
        Binary cross-entropy loss
        """
        # Avoid log(0) by clipping predictions
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def backward(self, X, y_true, y_pred):
        """
        Backpropagation - compute gradients
        """
        m = X.shape[0]  # Number of samples
        
        # Output layer gradients
        dz2 = y_pred - y_true.reshape(-1, 1)  # Derivative of loss w.r.t z2
        dW2 = (1/m) * dz2.T @ self.a1        # Gradient for W2
        db2 = (1/m) * np.sum(dz2, axis=0)    # Gradient for b2
        
        # Hidden layer gradients
        da1 = dz2 @ self.W2                  # Backpropagate to hidden layer
        dz1 = da1 * sigmoid_derivative(self.z1)  # Apply derivative of activation
        dW1 = (1/m) * dz1.T @ X              # Gradient for W1
        db1 = (1/m) * np.sum(dz1, axis=0)   # Gradient for b1
        
        return dW1, db1, dW2, db2
    
    def update_parameters(self, dW1, db1, dW2, db2):
        """
        Update parameters using gradient descent
        """
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
    
    def train_step(self, X, y):
        """
        Single training step
        """
        # Forward propagation
        y_pred = self.forward(X)
        
        # Compute loss
        loss = self.compute_loss(y, y_pred)
        
        # Backward propagation
        dW1, db1, dW2, db2 = self.backward(X, y, y_pred)
        
        # Update parameters
        self.update_parameters(dW1, db1, dW2, db2)
        
        # Compute accuracy
        predictions = (y_pred >= 0.5).astype(int).flatten()
        accuracy = np.mean(predictions == y)
        
        return loss, accuracy
    
    def predict(self, X):
        """
        Make predictions on new data
        """
        y_pred = self.forward(X)
        return (y_pred >= 0.5).astype(int).flatten()
    
    def predict_proba(self, X):
        """
        Get prediction probabilities
        """
        return self.forward(X).flatten()

print("✓ Neural Network class created successfully!")
print("Ready to train on the customer data.")

## Training the Network with Stochastic Gradient Descent

Now let's train our neural network on the customer data and watch it learn!

In [ ]:
# Prepare the data
X_customer = np.column_stack([x1_age, x2_income])
y_customer = y_purchase

# Split into training and testing sets
train_size = int(0.8 * len(X_customer))
indices = np.random.permutation(len(X_customer))

X_train = X_customer[indices[:train_size]]
y_train = y_customer[indices[:train_size]]
X_test = X_customer[indices[train_size:]]
y_test = y_customer[indices[train_size:]]

print(f"Training Data: {len(X_train)} samples")
print(f"Testing Data: {len(X_test)} samples")

# Create and train the network
network = SimpleNeuralNetwork(input_size=2, hidden_size=4, output_size=1, learning_rate=0.5)

# Training loop
epochs = 1000
batch_size = 32

print(f"\nTraining Neural Network...")
print(f"Epochs: {epochs}, Batch Size: {batch_size}, Learning Rate: {network.learning_rate}")

for epoch in range(epochs):
    # Shuffle training data
    shuffle_indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[shuffle_indices]
    y_train_shuffled = y_train[shuffle_indices]
    
    epoch_losses = []
    epoch_accuracies = []
    
    # Mini-batch training
    for i in range(0, len(X_train), batch_size):
        batch_X = X_train_shuffled[i:i+batch_size]
        batch_y = y_train_shuffled[i:i+batch_size]
        
        loss, accuracy = network.train_step(batch_X, batch_y)
        epoch_losses.append(loss)
        epoch_accuracies.append(accuracy)
    
    # Store average metrics for this epoch
    avg_loss = np.mean(epoch_losses)
    avg_accuracy = np.mean(epoch_accuracies)
    network.loss_history.append(avg_loss)
    network.accuracy_history.append(avg_accuracy)
    
    # Print progress every 100 epochs
    if (epoch + 1) % 100 == 0 or epoch < 10:
        # Test on validation set
        test_pred_proba = network.predict_proba(X_test)
        test_predictions = (test_pred_proba >= 0.5).astype(int)
        test_accuracy = np.mean(test_predictions == y_test)
        test_loss = network.compute_loss(y_test, test_pred_proba.reshape(-1, 1))
        
        print(f"Epoch {epoch+1:4d}: Train Loss={avg_loss:.4f}, Train Acc={avg_accuracy:.3f}, "
              f"Test Loss={test_loss:.4f}, Test Acc={test_accuracy:.3f}")

print(f"\n✓ Training completed!")

# Final evaluation
final_predictions = network.predict(X_test)
final_accuracy = np.mean(final_predictions == y_test)
print(f"\nFinal Test Accuracy: {final_accuracy:.1%}")

In [ ]:
# Visualize training progress and results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Training curves
axes[0, 0].plot(network.loss_history, 'b-', label='Loss', linewidth=2)
axes[0, 0].set_title('Training Loss Over Time')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(network.accuracy_history, 'g-', label='Accuracy', linewidth=2)
axes[0, 1].set_title('Training Accuracy Over Time')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Plot 2: Decision boundary
# Create a mesh to plot the decision boundary
h = 0.1
x_min, x_max = X_customer[:, 0].min() - 1, X_customer[:, 0].max() + 1
y_min, y_max = X_customer[:, 1].min() - 1, X_customer[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Get predictions for the entire mesh
mesh_points = np.c_[xx.ravel(), yy.ravel()]
Z = network.predict_proba(mesh_points)
Z = Z.reshape(xx.shape)

# Plot decision boundary
axes[1, 0].contourf(xx, yy, Z, levels=50, alpha=0.6, cmap='RdYlBu')
scatter = axes[1, 0].scatter(X_customer[:, 0], X_customer[:, 1], c=y_customer, 
                           cmap='RdYlBu', edgecolors='black', alpha=0.7)
axes[1, 0].set_title('Neural Network Decision Boundary')
axes[1, 0].set_xlabel('Age (normalized)')
axes[1, 0].set_ylabel('Income (normalized)')

# Add true decision boundary for comparison
x_line = np.linspace(x_min, x_max, 100)
y_line = -x_line
axes[1, 0].plot(x_line, y_line, 'k--', linewidth=3, label='True Boundary')
axes[1, 0].legend()

# Plot 3: Prediction probabilities histogram
test_probabilities = network.predict_proba(X_test)
axes[1, 1].hist(test_probabilities[y_test == 0], bins=20, alpha=0.7, 
               label='No Purchase', color='red', density=True)
axes[1, 1].hist(test_probabilities[y_test == 1], bins=20, alpha=0.7, 
               label='Purchase', color='blue', density=True)
axes[1, 1].axvline(x=0.5, color='black', linestyle='--', linewidth=2, 
                  label='Decision Threshold')
axes[1, 1].set_title('Prediction Probability Distribution')
axes[1, 1].set_xlabel('Predicted Probability')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final network parameters
print("\nFinal Network Parameters:")
print("Input → Hidden Weights:")
for i in range(network.W1.shape[0]):
    print(f"  Hidden {i+1}: Age={network.W1[i,0]:6.3f}, Income={network.W1[i,1]:6.3f}, Bias={network.b1[i]:6.3f}")

print("\nHidden → Output Weights:")
for i in range(network.W2.shape[1]):
    print(f"  From Hidden {i+1}: {network.W2[0,i]:6.3f}")
print(f"  Output Bias: {network.b2[0]:6.3f}")

# Analyze what the network learned
print(f"\nNetwork Analysis:")
print(f"• The network learned to approximate the rule: Age + Income ≥ 0")
print(f"• Final training accuracy: {network.accuracy_history[-1]:.1%}")
print(f"• Final test accuracy: {final_accuracy:.1%}")
print(f"• The decision boundary closely matches the true boundary (black dashed line)")

## Now: Scaling to Digit Recognition!

Great! You've seen how neural networks work on a simple 2D problem. Now let's apply the same principles to recognize handwritten digits - a much more challenging task with 64 input features instead of just 2!

### The Challenge
- **Input**: 8×8 pixel images (64 features) of handwritten digits
- **Output**: Classification into 10 classes (digits 0-9)
- **Network**: We'll use a larger hidden layer to handle the complexity

This is a real-world machine learning problem that showcases the power of neural networks!

In [ ]:
# Load the digits dataset
digits = load_digits()
X_digits = digits.data  # 8x8 images flattened to 64 features
y_digits = digits.target  # Digit labels 0-9

print(f"Digits Dataset:")
print(f"  Total samples: {X_digits.shape[0]:,}")
print(f"  Features per sample: {X_digits.shape[1]} (8×8 pixels)")
print(f"  Classes: {len(np.unique(y_digits))} (digits 0-9)")
print(f"  Feature range: [{X_digits.min():.1f}, {X_digits.max():.1f}]")

# Show some example digits
fig, axes = plt.subplots(2, 10, figsize=(15, 4))

for digit in range(10):
    # Find first occurrence of each digit
    idx = np.where(y_digits == digit)[0][0]
    
    # Original image
    axes[0, digit].imshow(X_digits[idx].reshape(8, 8), cmap='gray')
    axes[0, digit].set_title(f'Digit {digit}')
    axes[0, digit].axis('off')
    
    # Show another example
    idx2 = np.where(y_digits == digit)[0][1]  # Second occurrence
    axes[1, digit].imshow(X_digits[idx2].reshape(8, 8), cmap='gray')
    axes[1, digit].axis('off')

plt.suptitle('Sample Handwritten Digits (8×8 pixels)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Show class distribution
plt.figure(figsize=(10, 6))
unique, counts = np.unique(y_digits, return_counts=True)
plt.bar(unique, counts, color='skyblue', alpha=0.7)
plt.title('Distribution of Digits in Dataset')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.xticks(unique)
plt.grid(True, alpha=0.3)

for i, count in enumerate(counts):
    plt.text(i, count + 2, str(count), ha='center', fontweight='bold')

plt.show()

print(f"\nClass distribution is quite balanced - good for training!")

## Multi-Class Neural Network

For digit recognition, we need to modify our network for **multi-class classification** (10 classes instead of 2). The key changes:

1. **Output Layer**: 10 neurons (one for each digit)
2. **Activation**: Softmax instead of sigmoid (gives probability distribution)
3. **Loss Function**: Categorical cross-entropy
4. **Labels**: One-hot encoding (e.g., digit '3' becomes [0,0,0,1,0,0,0,0,0,0])

In [ ]:
def softmax(z):
    """Softmax activation for multi-class classification"""
    # Subtract max for numerical stability
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def one_hot_encode(y, num_classes):
    """Convert labels to one-hot encoding"""
    encoded = np.zeros((len(y), num_classes))
    encoded[np.arange(len(y)), y] = 1
    return encoded

class DigitClassifier:
    def __init__(self, input_size=64, hidden_size=100, output_size=10, learning_rate=0.01):
        self.learning_rate = learning_rate
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # Initialize weights with Xavier initialization
        self.W1 = np.random.normal(0, np.sqrt(2/input_size), (hidden_size, input_size))
        self.b1 = np.zeros((hidden_size,))
        self.W2 = np.random.normal(0, np.sqrt(2/hidden_size), (output_size, hidden_size))
        self.b2 = np.zeros((output_size,))
        
        # Training history
        self.loss_history = []
        self.accuracy_history = []
        
    def forward(self, X):
        """Forward propagation"""
        # Input to hidden (with ReLU activation for better performance)
        self.z1 = X @ self.W1.T + self.b1
        self.a1 = np.maximum(0, self.z1)  # ReLU activation
        
        # Hidden to output (with softmax)
        self.z2 = self.a1 @ self.W2.T + self.b2
        self.a2 = softmax(self.z2)
        
        return self.a2
    
    def compute_loss(self, y_true_onehot, y_pred):
        """Categorical cross-entropy loss"""
        # Avoid log(0)
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return -np.mean(np.sum(y_true_onehot * np.log(y_pred), axis=1))
    
    def backward(self, X, y_true_onehot, y_pred):
        """Backpropagation"""
        m = X.shape[0]
        
        # Output layer gradients
        dz2 = y_pred - y_true_onehot
        dW2 = (1/m) * dz2.T @ self.a1
        db2 = (1/m) * np.sum(dz2, axis=0)
        
        # Hidden layer gradients (ReLU derivative)
        da1 = dz2 @ self.W2
        dz1 = da1 * (self.z1 > 0)  # ReLU derivative
        dW1 = (1/m) * dz1.T @ X
        db1 = (1/m) * np.sum(dz1, axis=0)
        
        return dW1, db1, dW2, db2
    
    def update_parameters(self, dW1, db1, dW2, db2):
        """Update parameters with gradient descent"""
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
    
    def train_step(self, X, y):
        """Single training step"""
        y_onehot = one_hot_encode(y, self.output_size)
        
        # Forward propagation
        y_pred = self.forward(X)
        
        # Compute loss
        loss = self.compute_loss(y_onehot, y_pred)
        
        # Backward propagation
        dW1, db1, dW2, db2 = self.backward(X, y_onehot, y_pred)
        
        # Update parameters
        self.update_parameters(dW1, db1, dW2, db2)
        
        # Compute accuracy
        predictions = np.argmax(y_pred, axis=1)
        accuracy = np.mean(predictions == y)
        
        return loss, accuracy
    
    def predict(self, X):
        """Make predictions"""
        y_pred = self.forward(X)
        return np.argmax(y_pred, axis=1)
    
    def predict_proba(self, X):
        """Get prediction probabilities"""
        return self.forward(X)

print("✓ Multi-class Neural Network created!")
print("Key differences from binary classification:")
print("  • ReLU activation in hidden layer (better for deep networks)")
print("  • Softmax activation in output layer (probability distribution)")
print("  • Categorical cross-entropy loss")
print("  • One-hot encoded targets")

## Training the Digit Classifier

Now let's train our neural network to recognize handwritten digits!

In [ ]:
# Prepare the digits data
# Normalize pixel values to [0, 1] range
X_digits_normalized = X_digits / 16.0  # Max pixel value is 16

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_digits_normalized, y_digits, test_size=0.2, random_state=42, stratify=y_digits
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Input features: {X_train.shape[1]} (8×8 pixels)")
print(f"Classes: {len(np.unique(y_train))}")

# Create and train the digit classifier
classifier = DigitClassifier(
    input_size=64,      # 8×8 pixels
    hidden_size=128,    # More neurons for complex patterns
    output_size=10,     # 10 digit classes
    learning_rate=0.1   # Learning rate
)

# Training parameters
epochs = 500
batch_size = 32

print(f"\nTraining Digit Classifier...")
print(f"Architecture: {classifier.input_size} → {classifier.hidden_size} → {classifier.output_size}")
print(f"Parameters: {classifier.W1.size + classifier.b1.size + classifier.W2.size + classifier.b2.size:,} total")
print(f"Training: {epochs} epochs, batch size {batch_size}")

# Training loop with progress tracking
best_accuracy = 0
patience = 50
patience_counter = 0

for epoch in range(epochs):
    # Shuffle training data
    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]
    
    epoch_losses = []
    epoch_accuracies = []
    
    # Mini-batch training
    for i in range(0, len(X_train), batch_size):
        batch_X = X_train_shuffled[i:i+batch_size]
        batch_y = y_train_shuffled[i:i+batch_size]
        
        loss, accuracy = classifier.train_step(batch_X, batch_y)
        epoch_losses.append(loss)
        epoch_accuracies.append(accuracy)
    
    # Store average metrics
    avg_loss = np.mean(epoch_losses)
    avg_accuracy = np.mean(epoch_accuracies)
    classifier.loss_history.append(avg_loss)
    classifier.accuracy_history.append(avg_accuracy)
    
    # Evaluate on test set every 25 epochs
    if (epoch + 1) % 25 == 0 or epoch < 10:
        test_predictions = classifier.predict(X_test)
        test_accuracy = np.mean(test_predictions == y_test)
        test_probabilities = classifier.predict_proba(X_test)
        test_loss = classifier.compute_loss(
            one_hot_encode(y_test, 10), test_probabilities
        )
        
        print(f"Epoch {epoch+1:3d}: Train Loss={avg_loss:.4f}, Train Acc={avg_accuracy:.3f}, "
              f"Test Loss={test_loss:.4f}, Test Acc={test_accuracy:.3f}")
        
        # Early stopping
        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            patience_counter = 0
        else:
            patience_counter += 25
            
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break

print(f"\n✓ Training completed!")
print(f"Best test accuracy: {best_accuracy:.1%}")

# Final evaluation
final_predictions = classifier.predict(X_test)
final_accuracy = np.mean(final_predictions == y_test)
print(f"Final test accuracy: {final_accuracy:.1%}")

## Results Analysis and Visualization

Let's analyze how well our neural network learned to recognize digits!

In [ ]:
# Comprehensive results analysis
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

# Plot 1: Training curves
axes[0, 0].plot(classifier.loss_history, 'b-', linewidth=2, label='Loss')
axes[0, 0].set_title('Training Loss Over Time')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(classifier.accuracy_history, 'g-', linewidth=2, label='Accuracy')
axes[0, 1].set_title('Training Accuracy Over Time')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Plot 2: Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, final_predictions)
im = axes[0, 2].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0, 2].set_title('Confusion Matrix')
axes[0, 2].set_xlabel('Predicted Digit')
axes[0, 2].set_ylabel('True Digit')

# Add text annotations to confusion matrix
for i in range(10):
    for j in range(10):
        text = axes[0, 2].text(j, i, cm[i, j], ha="center", va="center",
                              color="white" if cm[i, j] > cm.max() / 2 else "black")

# Plot 3: Per-class accuracy
class_accuracies = []
for digit in range(10):
    mask = y_test == digit
    if np.sum(mask) > 0:
        acc = np.mean(final_predictions[mask] == digit)
        class_accuracies.append(acc)
    else:
        class_accuracies.append(0)

bars = axes[1, 0].bar(range(10), class_accuracies, color='skyblue', alpha=0.7)
axes[1, 0].set_title('Accuracy by Digit Class')
axes[1, 0].set_xlabel('Digit')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_xticks(range(10))
axes[1, 0].grid(True, alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars, class_accuracies):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{acc:.2f}', ha='center', va='bottom', fontsize=9)

# Plot 4: Correct predictions examples
correct_mask = final_predictions == y_test
correct_indices = np.where(correct_mask)[0][:15]  # First 15 correct

axes[1, 1].set_title('Examples of Correct Predictions')
for i, idx in enumerate(correct_indices[:15]):
    row, col = i // 5, i % 5
    if row < 3:
        ax_pos = [0.4 + col*0.12, 0.65 - row*0.08, 0.08, 0.06]
        ax_img = fig.add_axes(ax_pos)
        ax_img.imshow(X_test[idx].reshape(8, 8), cmap='gray')
        ax_img.set_title(f'{y_test[idx]}', fontsize=8)
        ax_img.axis('off')

axes[1, 1].axis('off')

# Plot 5: Incorrect predictions examples
incorrect_mask = final_predictions != y_test
incorrect_indices = np.where(incorrect_mask)[0][:15]  # First 15 incorrect

axes[1, 2].set_title('Examples of Incorrect Predictions')
for i, idx in enumerate(incorrect_indices[:15]):
    row, col = i // 5, i % 5
    if row < 3:
        ax_pos = [0.68 + col*0.12, 0.65 - row*0.08, 0.08, 0.06]
        ax_img = fig.add_axes(ax_pos)
        ax_img.imshow(X_test[idx].reshape(8, 8), cmap='gray')
        ax_img.set_title(f'{y_test[idx]}→{final_predictions[idx]}', fontsize=8)
        ax_img.axis('off')

axes[1, 2].axis('off')

# Plot 6: Prediction confidence distribution
test_probabilities = classifier.predict_proba(X_test)
max_probabilities = np.max(test_probabilities, axis=1)

axes[2, 0].hist(max_probabilities[correct_mask], bins=30, alpha=0.7, 
               label='Correct', color='green', density=True)
if np.sum(incorrect_mask) > 0:
    axes[2, 0].hist(max_probabilities[incorrect_mask], bins=30, alpha=0.7, 
                   label='Incorrect', color='red', density=True)
axes[2, 0].set_title('Prediction Confidence Distribution')
axes[2, 0].set_xlabel('Max Probability (Confidence)')
axes[2, 0].set_ylabel('Density')
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

# Plot 7: Learning curve analysis
window_size = 20
if len(classifier.accuracy_history) > window_size:
    smoothed_acc = np.convolve(classifier.accuracy_history, 
                              np.ones(window_size)/window_size, mode='valid')
    axes[2, 1].plot(range(window_size-1, len(classifier.accuracy_history)), 
                   smoothed_acc, 'b-', linewidth=2, label='Smoothed Accuracy')
    axes[2, 1].plot(classifier.accuracy_history, 'b-', alpha=0.3, label='Raw Accuracy')
else:
    axes[2, 1].plot(classifier.accuracy_history, 'b-', linewidth=2, label='Accuracy')

axes[2, 1].set_title('Learning Progress (Smoothed)')
axes[2, 1].set_xlabel('Epoch')
axes[2, 1].set_ylabel('Accuracy')
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

# Plot 8: Network weights visualization
# Show first layer weights as images
weights_sample = classifier.W1[:16]  # First 16 hidden neurons
weights_grid = weights_sample.reshape(16, 8, 8)

# Create a 4x4 grid of weight visualizations
weight_mosaic = np.zeros((4*8, 4*8))
for i in range(4):
    for j in range(4):
        idx = i*4 + j
        weight_mosaic[i*8:(i+1)*8, j*8:(j+1)*8] = weights_grid[idx]

im = axes[2, 2].imshow(weight_mosaic, cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[2, 2].set_title('First Layer Weights\n(16 Hidden Neurons)')
axes[2, 2].axis('off')

plt.tight_layout()
plt.show()

# Print detailed analysis
print("\n" + "="*60)
print("NEURAL NETWORK ANALYSIS RESULTS")
print("="*60)

print(f"\nOverall Performance:")
print(f"  • Final accuracy: {final_accuracy:.1%}")
print(f"  • Total parameters: {classifier.W1.size + classifier.b1.size + classifier.W2.size + classifier.b2.size:,}")
print(f"  • Training epochs: {len(classifier.loss_history)}")

print(f"\nPer-Class Performance:")
for digit in range(10):
    print(f"  Digit {digit}: {class_accuracies[digit]:.1%} accuracy")

print(f"\nError Analysis:")
total_errors = np.sum(incorrect_mask)
print(f"  • Total errors: {total_errors}/{len(y_test)} ({total_errors/len(y_test):.1%})")
if total_errors > 0:
    avg_confidence_correct = np.mean(max_probabilities[correct_mask])
    avg_confidence_incorrect = np.mean(max_probabilities[incorrect_mask])
    print(f"  • Average confidence on correct predictions: {avg_confidence_correct:.1%}")
    print(f"  • Average confidence on incorrect predictions: {avg_confidence_incorrect:.1%}")

print(f"\nModel Insights:")
print(f"  • The network learned meaningful digit patterns")
print(f"  • Hidden layer weights show feature detectors for edges and shapes")
print(f"  • High confidence on correct predictions indicates good calibration")
print(f"  • Performance is competitive with traditional machine learning methods")

## Congratulations! 🎉

You've successfully completed the Python Workshop capstone project! Here's what you accomplished:

### What You Built
- **From scratch neural network**: No black-box libraries - you implemented every component
- **Multi-class classifier**: Handles 10 different digit classes
- **Real-world performance**: Achieved competitive accuracy on a classic ML problem
- **Professional analysis**: Created comprehensive visualizations and interpretations

### Skills Demonstrated
✅ **NumPy mastery**: Matrix operations, broadcasting, mathematical functions  
✅ **Algorithm implementation**: Forward/backward propagation, gradient descent  
✅ **Data preprocessing**: Normalization, train/test splits, one-hot encoding  
✅ **Visualization**: Training curves, confusion matrices, prediction analysis  
✅ **Code organization**: Classes, functions, modular design  
✅ **Problem solving**: Debugging, hyperparameter tuning, performance optimization  

### From Simple to Sophisticated
You started with a 2-input, 3-hidden-neuron network learning `x₁ + x₂ ≥ 0` and scaled to a 64-input, 128-hidden-neuron network recognizing handwritten digits. This progression demonstrates the power and scalability of neural networks!

### Real-World Impact
The techniques you've learned apply to:
- **Computer vision**: Image recognition, medical imaging, autonomous vehicles
- **Natural language processing**: Text classification, sentiment analysis
- **Business analytics**: Customer segmentation, fraud detection, recommendation systems
- **Scientific computing**: Pattern recognition in research data

### Next Steps
You're now equipped to:
1. **Explore deep learning frameworks** like TensorFlow or PyTorch
2. **Tackle larger datasets** with convolutional or recurrent neural networks
3. **Apply to your domain** using the same principles you've mastered
4. **Continue learning** advanced topics like regularization, optimization, and architecture design

**Well done!** You've gone from Python basics to implementing state-of-the-art machine learning algorithms. This is a remarkable achievement that opens doors to exciting opportunities in data science, AI, and beyond!

## Bonus Challenge: Experiment and Improve!

Ready for more? Try these extensions to deepen your understanding:

### Easy Experiments
1. **Change the architecture**: Try different hidden layer sizes (50, 200, 500 neurons)
2. **Adjust learning rate**: Test values like 0.01, 0.1, 1.0 and observe the impact
3. **Add more layers**: Create a deeper network with 2-3 hidden layers

### Medium Challenges
4. **Implement different activations**: Try tanh or leaky ReLU instead of ReLU
5. **Add regularization**: Implement dropout or L2 regularization to prevent overfitting
6. **Batch normalization**: Normalize inputs to each layer for faster training

### Advanced Projects
7. **Convolutional layers**: Add 2D convolutions for better image understanding
8. **Different datasets**: Try CIFAR-10 (color images) or fashion-MNIST
9. **Ensemble methods**: Combine multiple networks for better performance

### Research Questions
10. **Visualization**: What patterns do different hidden neurons detect?
11. **Adversarial examples**: Can you fool the network with small input changes?
12. **Transfer learning**: Use pre-trained weights from one problem on another

Each experiment will teach you more about how neural networks learn and how to optimize them for real-world applications!